# 04 · Insight dan visualisasi

**Analisis awal, bukan solusi final.** Notebook ini hanya membaca hasil `03_processing` dan
menampilkannya sebagai grafik, masing-masing dengan satu kalimat tentang artinya untuk bisnis.

Semua angka berasal dari toko yang kebetulan terbaca dalam satu run. Baca sebagai petunjuk
arah, bukan sebagai ukuran pasar.

In [ ]:
import matplotlib.pyplot as plt
import nbutil as nb
import numpy as np
import pandas as pd
from matplotlib.patches import Patch

nb.use_style()
print("Run:", nb.run_id())
stores = nb.load("stores_clean")
ranked = nb.load("brands_ranked")
overlap = nb.load("brand_overlap")
jaccard = nb.load("brand_jaccard").set_index("key")
opportunities = nb.load("opportunities")
regions = nb.load("regions")
name_of = ranked.set_index("brand_key")["brand"].to_dict() | {
    "woodenships": "Wooden Ships (pelanggan)"
}
KIND_COLOR = {"pesaing knitwear": nb.SERIES[0], "brand penanda": nb.SERIES[1]}


def label_bars(ax, values, offset, fmt="{:.0f}"):
    for y, value in enumerate(values):
        ax.text(value + offset, y, fmt.format(value), va="center", color=nb.INK_2, fontsize=8.5)

## 1. Dari semua toko ke daftar peluang

Corong ini menunjukkan di mana toko "hilang". Kehilangan terbesar ada di langkah pertama:
toko yang tidak terbaca.

In [ ]:
funnel = pd.Series(
    {
        "Semua toko dalam run": len(stores),
        "Katalognya terbaca": int(stores["readable"].sum()),
        "Kandidat partner, belum pelanggan": len(opportunities),
        "Menjual minimal satu brand relevan": int((opportunities["n_relevant"] > 0).sum()),
        "Di antaranya punya email": int(
            ((opportunities["n_relevant"] > 0) & opportunities["has_email"]).sum()
        ),
    }
)[::-1]
fig, ax = plt.subplots(figsize=(7, 2.6))
ax.barh(funnel.index, funnel.values, height=0.55, color=nb.SERIES[0])
label_bars(ax, funnel.values, 8)
ax.set_title("Dari toko yang ditemukan sampai toko yang bisa dihubungi")
ax.set_xlabel("jumlah toko")
ax.xaxis.grid(True)
ax.set_axisbelow(True)
plt.show()

**Artinya:** menambah toko yang terbaca (lihat uji baca ulang toko `blocked`) menaikkan
semua angka di bawahnya sekaligus. Itu lebih murah daripada mencari toko baru.

## 2. Brand yang paling banyak dijual, dan jenisnya

Warna membedakan dua jenis brand: **pesaing knitwear** (porsi knitwear tinggi) dan **brand
penanda** (populer di butik sasaran, tetapi knitwear hanya sebagian kecil produknya).

In [ ]:
top = ranked[ranked["enough_data"]].nlargest(20, "stores").iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 5.2))
ax.barh(top["brand"], top["stores"], height=0.6, color=top["kind"].map(KIND_COLOR))
for y, (count, share) in enumerate(zip(top["stores"], top["knit_share"], strict=True)):
    ax.text(
        count + 0.4,
        y,
        f"{count} toko · knitwear {share:.0%}",
        va="center",
        color=nb.INK_2,
        fontsize=8,
    )
ax.set_title("20 brand yang terlihat di paling banyak toko")
ax.set_xlabel("jumlah toko yang menjualnya")
ax.set_xlim(0, top["stores"].max() * 1.45)
ax.xaxis.grid(True)
ax.set_axisbelow(True)
ax.legend(handles=[Patch(color=c, label=k) for k, c in KIND_COLOR.items()], loc="lower right")
plt.show()

**Artinya:** brand yang paling populer sebagian besar adalah brand penanda. Mereka berguna
untuk menemukan butik yang tepat, tetapi bukan mereka yang memperebutkan rak knitwear.

## 3. Peta brand: seberapa populer dan seberapa "knitwear"

Setiap titik adalah satu brand dengan cukup data. Kanan atas adalah brand yang paling perlu
diperhatikan: banyak dijual dan sebagian besar produknya knitwear. Hanya brand yang dipantau
dan brand paling kanan yang diberi nama; sisanya ada di tabel `brands_ranked`.

In [ ]:
plot = ranked[ranked["enough_data"]]
fig, ax = plt.subplots(figsize=(7, 4.4))
for kind, color in KIND_COLOR.items():
    part = plot[plot["kind"] == kind]
    ax.scatter(
        part["knit_share"],
        part["stores"],
        s=34,
        color=color,
        edgecolor="white",
        linewidth=1,
        label=kind,
        zorder=3,
    )
label = plot[plot["watched"] | (plot["knit_share"] >= 0.8)].sort_values("knit_share")
for i, (_, row) in enumerate(label.iterrows()):
    offset = (5, 5) if i % 2 == 0 else (5, -11)  # alternate, so neighbours do not collide
    ax.annotate(
        row["brand"],
        (row["knit_share"], row["stores"]),
        xytext=offset,
        textcoords="offset points",
        fontsize=7.5,
        color=nb.INK_2,
    )
ax.axvline(0.25, color=nb.INK_3, linewidth=0.8)
ax.text(0.255, ax.get_ylim()[1] * 0.97, "batas 25%", color=nb.INK_3, fontsize=7.5, va="top")
ax.set_title("Popularitas brand dibanding porsi knitwear-nya")
ax.set_xlabel("porsi produk yang knitwear")
ax.set_ylabel("jumlah toko yang menjualnya")
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.grid(True)
ax.set_axisbelow(True)
ax.legend(loc="upper right")
plt.show()

**Artinya:** daftar brand yang dipantau sebaiknya dipecah dua. Kelompok kanan adalah pesaing
langsung. Kelompok kiri atas adalah penanda tipe butik.

## 4. Posisi harga brand terhadap Wooden Ships

Nilai 1,0 berarti median harga knitwear brand itu setara dengan harga acuan kita. Bidang
abu-abu adalah rentang yang dianggap "cocok" oleh aturan sekarang.

Harga acuan masih diambil dari sedikit produk Wooden Ships di katalog stockist, jadi grafik
ini harus dihitung ulang setelah harga dari Shopify kita dimasukkan.

In [ ]:
priced = ranked[ranked["price_position"].notna() & (ranked["enough_data"] | ranked["watched"])]
show = pd.concat(
    [priced[priced["kind"] == "pesaing knitwear"].nlargest(14, "stores"), priced[priced["watched"]]]
).drop_duplicates("brand_key")
show = show.sort_values("price_position")
fig, ax = plt.subplots(figsize=(7, 0.3 * len(show) + 1.2))
ax.axvspan(0.75, 1.35, color="#eeedea", zorder=0)
ax.axvline(1.0, color=nb.INK_3, linewidth=0.8, zorder=1)
ax.hlines(show["brand"], 1.0, show["price_position"], color=nb.INK_3, linewidth=0.8, zorder=2)
ax.scatter(
    show["price_position"],
    show["brand"],
    s=42,
    color=show["kind"].map(KIND_COLOR),
    edgecolor="white",
    linewidth=1,
    zorder=3,
)
for brand, value in zip(show["brand"], show["price_position"], strict=True):
    side = 1 if value >= 1 else -1  # label on the outer side of the dot, clear of its line
    ax.text(
        value + 0.05 * side,
        brand,
        f"{value:.2f}×",
        va="center",
        ha="left" if side > 0 else "right",
        color=nb.INK_2,
        fontsize=8,
    )
ax.set_title("Median harga knitwear brand dibanding harga acuan Wooden Ships")
ax.set_xlabel("kali harga acuan kita (USD saja)")
ax.set_xlim(0, max(3.0, show["price_position"].max() * 1.15))
ax.legend(handles=[Patch(color=c, label=k) for k, c in KIND_COLOR.items()], loc="lower right")
plt.show()

**Artinya:** brand dengan porsi knitwear tinggi belum tentu sekelas harga dengan kita. Brand
yang jauh di kanan menjual di tingkat harga lain, jadi butiknya belum tentu cocok.

## 5. Brand mana yang distribusinya mirip

Angka di tiap kotak adalah porsi toko yang sama di antara dua brand (Jaccard). Makin gelap,
makin sering keduanya dijual di toko yang sama.

In [ ]:
keys = ranked[ranked["enough_data"]].nlargest(12, "stores")["brand_key"].tolist()
keys += [
    k for k in ranked.loc[ranked["watched"], "brand_key"] if k not in keys and k in jaccard.index
]
keys += ["woodenships"]
grid = jaccard.loc[keys, keys].to_numpy().copy()
np.fill_diagonal(grid, np.nan)
labels = [name_of[k] for k in keys]

fig, ax = plt.subplots(figsize=(7.4, 6.2))
image = ax.imshow(grid, cmap="Blues", vmin=0, vmax=np.nanmax(grid))
ax.set_xticks(range(len(keys)), labels, rotation=45, ha="right")
ax.set_yticks(range(len(keys)), labels)
for i in range(len(keys)):
    for j in range(len(keys)):
        if i != j:
            ax.text(
                j,
                i,
                f"{grid[i, j]:.2f}"[1:],
                ha="center",
                va="center",
                fontsize=7,
                color="white" if grid[i, j] > 0.6 * np.nanmax(grid) else nb.INK,
            )
for side in ("left", "bottom"):
    ax.spines[side].set_visible(False)
ax.tick_params(length=0)
ax.set_title("Porsi toko yang sama di antara dua brand")
fig.colorbar(image, ax=ax, shrink=0.6, label="Jaccard")
plt.show()

**Artinya:** brand yang saling gelap membentuk "kelompok butik" yang sama. Baris Wooden Ships
masih sangat tipis, karena hanya sedikit pelanggan kita yang ada di antara toko terbaca.

## 6. Brand yang dipantau: di mana tokonya terhadap kita

Untuk tiap brand yang dipantau, toko yang menjualnya dibagi empat: sudah pelanggan kita,
dekat stockist kita, di wilayah kosong, atau lokasinya belum diketahui.

In [ ]:
watched = overlap[
    overlap["brand_key"].isin(ranked.loc[ranked["watched"], "brand_key"])
].sort_values("stores")
segments = {
    "Sudah pelanggan kita": "shared",
    "Bukan kita, ada stockist dalam radius": "near_stockist",
    "Bukan kita, wilayah kosong": "open_territory",
    "Bukan kita, lokasi tidak diketahui": "location_unknown",
}
colors = [nb.SERIES[0], nb.SERIES[1], nb.SERIES[2], "#b8b6ae"]
fig, ax = plt.subplots(figsize=(7, 0.55 * len(watched) + 1.4))
left = np.zeros(len(watched))
for (label, column), color in zip(segments.items(), colors, strict=True):
    values = watched[column].to_numpy()
    ax.barh(
        watched["brand"],
        values,
        left=left,
        height=0.5,
        color=color,
        edgecolor="white",
        linewidth=2,
        label=label,
    )
    for y, (start, value) in enumerate(zip(left, values, strict=True)):
        if value >= 2:
            ax.text(
                start + value / 2,
                y,
                f"{value}",
                ha="center",
                va="center",
                fontsize=8,
                color="white" if color != "#b8b6ae" else nb.INK,
            )
    left += values
ax.set_title("Toko yang menjual brand yang dipantau")
ax.set_xlabel("jumlah toko")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.32), ncol=2)
plt.show()

**Artinya:** hampir semua toko yang menjual brand ini belum menjadi pelanggan kita. Bagian
"wilayah kosong" adalah yang bisa dihubungi tanpa bentrok teritori. Bagian "lokasi tidak
diketahui" perlu dicari alamatnya dulu.

## 7. Toko peluang menurut jumlah brand relevan yang dijual

Makin banyak brand relevan yang dijual sebuah toko, makin kuat buktinya bahwa toko itu
membeli di kategori kita.

In [ ]:
buckets = pd.cut(
    opportunities["n_relevant"],
    [-1, 0, 1, 2, 4, 100],
    labels=["0", "1", "2", "3–4", "5 atau lebih"],
)
table = pd.crosstab(buckets, opportunities["territory"]).reindex(
    columns=["wilayah kosong", "ada stockist dalam radius", "lokasi tidak diketahui"], fill_value=0
)
fig, ax = plt.subplots(figsize=(7, 3))
bottom = np.zeros(len(table))
for column, color in zip(table.columns, [nb.SERIES[2], nb.SERIES[1], "#b8b6ae"], strict=True):
    ax.bar(
        table.index.astype(str),
        table[column],
        bottom=bottom,
        width=0.5,
        color=color,
        edgecolor="white",
        linewidth=2,
        label=column,
    )
    bottom += table[column].to_numpy()
for x, total in enumerate(bottom):
    ax.text(x, total + 1.5, f"{int(total)}", ha="center", color=nb.INK_2, fontsize=8.5)
ax.set_ylim(0, bottom.max() * 1.15)
ax.set_title("Kandidat partner menurut jumlah brand relevan yang dijualnya")
ax.set_xlabel("jumlah brand relevan di katalog toko")
ax.set_ylabel("jumlah toko")
ax.yaxis.grid(True)
ax.set_axisbelow(True)
ax.legend(loc="upper right")
plt.show()

**Artinya:** mulai dari kanan. Toko dengan beberapa brand relevan di wilayah kosong adalah
prioritas pertama tim wholesale.

## 8. Wilayah

Kiri: toko peluang dari run ini per negara bagian, dibanding stockist kita. Kanan: contoh
daftar resmi sebuah brand, bila berkasnya ada. Daftar resmi jauh lebih lengkap daripada
sampel dari run.

In [ ]:
def paired(ax, frame, label_col, a, b, a_label, b_label, title):
    frame = frame.iloc[::-1]
    y = np.arange(len(frame))
    ax.barh(y + 0.2, frame[a], height=0.36, color=nb.SERIES[0], label=a_label)
    ax.barh(y - 0.2, frame[b], height=0.36, color=nb.SERIES[1], label=b_label)
    for yy, (va, vb) in enumerate(zip(frame[a], frame[b], strict=True)):
        ax.text(va + 0.6, yy + 0.2, f"{va}", va="center", fontsize=7.5, color=nb.INK_2)
        ax.text(vb + 0.6, yy - 0.2, f"{vb}", va="center", fontsize=7.5, color=nb.INK_2)
    ax.set_yticks(y, frame[label_col])
    ax.set_title(title)
    ax.set_xlabel("jumlah toko")
    ax.xaxis.grid(True)
    ax.set_axisbelow(True)
    ax.legend(loc="lower right")


locator_file = nb.out_dir() / "locator_states.parquet"
panels = 2 if locator_file.exists() else 1
fig, axes = plt.subplots(1, panels, figsize=(7.2 * panels / 1.25, 4.2), squeeze=False)
paired(
    axes[0][0],
    regions.head(10),
    "state",
    "opportunities",
    "stockists",
    "Toko peluang (sampel run)",
    "Stockist Wooden Ships",
    "Toko peluang dan stockist per negara bagian",
)
if locator_file.exists():
    states = pd.read_parquet(locator_file)
    brand = states["brand"].iloc[0]
    one = states[states["brand"] == brand].nlargest(10, "brand_retailers")
    paired(
        axes[0][1],
        one,
        "state",
        "brand_retailers",
        "stockists",
        f"Retailer {brand} (daftar resmi)",
        "Stockist Wooden Ships",
        f"{brand} dan stockist per negara bagian",
    )
fig.tight_layout()
plt.show()

**Artinya:** celah wilayah baru bisa dibaca dengan yakin dari daftar resmi. Dari sampel run,
jumlah toko per negara bagian masih terlalu kecil untuk menyimpulkan apa pun.

## Ringkasan insight

Dicetak dari data yang sedang dimuat.

In [ ]:
rivals = ranked[ranked["enough_data"] & (ranked["kind"] == "pesaing knitwear")]
relevant = opportunities[opportunities["n_relevant"] > 0]
print(
    f"1. {int(stores['readable'].sum())} dari {len(stores)} toko terbaca. Menambah toko terbaca adalah tuas terbesar."
)
print(
    f"2. {len(opportunities)} kandidat partner belum pelanggan; {len(relevant)} menjual brand relevan, {int(relevant['has_email'].sum())} di antaranya punya email."
)
print(
    f"3. {len(rivals)} brand lolos sebagai pesaing knitwear (≥8 toko, ≥30 produk, knitwear ≥25%)."
)
print(
    "   Lima teratas menurut peringkat sementara:",
    ", ".join(rivals.nlargest(5, "rank_score")["brand"]),
)
print("4. Brand yang dipantau:")
for _, row in ranked[ranked["watched"]].iterrows():
    print(
        f"   {row['brand']}: {row['stores']} toko, knitwear {row['knit_share']:.0%}, harga {row['price_position']:.2f}× acuan, {row['kind']}"
    )
strong = relevant[relevant["n_relevant"] >= 3]
print(
    f"5. {len(strong)} toko menjual tiga brand relevan atau lebih; {int((strong['territory'] == 'wilayah kosong').sum())} di wilayah kosong."
)
print(
    f"6. Lokasi belum diketahui untuk {int((opportunities['territory'] == 'lokasi tidak diketahui').sum())} dari {len(opportunities)} kandidat."
)

## Sebelum angka ini dipakai untuk keputusan

1. Baca ulang toko yang tadinya `blocked`, lalu jalankan keempat notebook lagi.
2. Masukkan harga dari katalog Shopify kita (`price_points.csv`), supaya posisi harga benar.
3. Masukkan akun Salesforce (`accounts.csv`), supaya "belum pelanggan" benar.
4. Minta tim wholesale menandai sekitar 30 brand, untuk menguji peringkat brand.
5. Putuskan arti rentang harga "50–60%" (PRD, Q-10).